# 02 · Data pipeline: documents → token shards → batches

The trainer reads **token shards**: flat `uint16` arrays with an EOS token between documents.
A training example is a contiguous window of `seq_len + 1` tokens; batches are drawn deterministically
from a seeded permutation, so a resumed job continues on exactly the batch it would have seen.

The class corpus is built once with `python -m gw1b.prepare_data tokenize ...` (a CPU batch job, see
`slurm/tokenize.sbatch`). Here we build a miniature version to see the moving parts.

In [ ]:
# --- GW1B setup cell (same in every notebook) -------------------------------------------------
import os, sys, json, time, pathlib
GW1B_HOME    = os.environ.get("GW1B_HOME", "/SEAS/groups/gw1b/gw1b-f2026")
GW1B_SCRATCH = os.environ.get("GW1B_SCRATCH", "/lustre/groups/gw1b")
GW1B_GROUP   = os.environ.get("GW1B_GROUP", "/SEAS/groups/gw1b")
USER_DIR     = os.path.join(GW1B_SCRATCH, "users", os.environ.get("USER", "student"))
os.makedirs(USER_DIR, exist_ok=True)
if GW1B_HOME not in sys.path:
    sys.path.insert(0, GW1B_HOME)        # the shared gw1b package
import jax, jax.numpy as jnp, numpy as np
print("JAX", jax.__version__, "| backend:", jax.default_backend(), "| devices:", jax.devices())
print("your scratch dir:", USER_DIR)


## 1. Tokenize documents into shards

In [ ]:
from gw1b.tokenizer import Tokenizer
from gw1b.prepare_data import tokenize_documents
tok_path = os.path.join(USER_DIR, "tok-bpe-4k.model")
if not os.path.exists(tok_path):
    print("run 01_tokenizer.ipynb first (it trains tok-bpe-4k) — using the class tokenizer instead")
    tok_path = os.path.join(GW1B_GROUP, "tokenizer", "gw1b-32k.model")
sample_path = os.path.join(USER_DIR, "tokenizer_sample.txt")
docs = (line.strip() for line in open(sample_path) if line.strip())
out_dir = os.path.join(USER_DIR, "data", "mini")
manifest = tokenize_documents(docs, tok_path, out_dir, val_docs=1000, workers=4, shard_size=200_000, chunk_docs=200)
{k: manifest[k] for k in ["documents_seen", "train_tokens", "val_tokens", "train_shards", "val_shards"]}

## 2. Look inside a shard

In [ ]:
from gw1b.data import TokenDataset, BatchLoader
tok = Tokenizer(tok_path)
ds = TokenDataset(os.path.join(out_dir, "train"), seq_len=128)
print(f"{ds.total_tokens:,} tokens in {len(ds.files)} shard(s) -> {ds.n_windows:,} windows of 128")
w = ds.window(0)
print("window 0 ids:", w[:20], "...")
print("decoded    :", repr(tok.decode(w[:60].tolist())))
print("EOS id", tok.eos_id, "appears", int((w == tok.eos_id).sum()), "times in this window (document boundaries)")

## 3. Deterministic, resumable batches

In [ ]:
loader = BatchLoader(ds, batch_size=8, seed=0, start_step=0)
b0, b1 = next(loader), next(loader); loader.close()
loader2 = BatchLoader(ds, batch_size=8, seed=0, start_step=1)     # "resume" at step 1
b1_again = next(loader2); loader2.close()
print("inputs shape", b0["inputs"].shape, "| targets are inputs shifted by one:", bool((b0["inputs"][:, 1:] == b0["targets"][:, :-1]).all()))
print("resumed batch identical:", bool((b1["inputs"] == b1_again["inputs"]).all()))
# in a multi-GPU job, each process gets its slice of the same global batch:
parts = []
for rank in range(4):
    l = BatchLoader(ds, batch_size=8, seed=0, rank=rank, world=4); parts.append(next(l)["inputs"]); l.close()
print("4 ranks concatenate to the global batch:", bool((np.concatenate(parts) == b0["inputs"]).all()))

## 4. Where Team 1 plugs in
* `tokenize_documents(docs, ...)` accepts **any iterator of strings** — your filtered / re-mixed / curriculum-ordered corpus.
* `--filter my_module:keep_fn` applies a document filter inside the tokenization workers.
* The `manifest.json` next to the shards records the tokenizer hash and source files = the *data manifest* we release.

Next: `03_train_proxy_model.ipynb`.